In [ ]:
import pandas as pd
from sklearn.metrics import matthews_corrcoef
import numpy as np
print('Libraries loaded')

In [ ]:
# Load the data for all models into a list of dataframes
model_files = [
    'gpt_output/GPT1_evaluate.tsv',
    'gpt_output/GPT2_evaluate.tsv',
    'gpt_output/GPT3_evaluate.tsv',
    'bert_output/bert1_evaluate.tsv',
    'bert_output/bert2_evaluate.tsv',
    'bert_output/bert3_evaluate.tsv'
]

# Descriptive model names
model_names = [
    'gpt2_model1',
    'gpt2_model2',
    'gpt2_model3',
    'bert_model1',
    'bert_model2',
    'bert_model3'
]

# Read all dataframes
dataframes = [pd.read_csv(file, sep='\t') for file in model_files]

# Convert predicted labels to numerical (e.g. LABEL_1 -> 1, LABEL_0 -> 0)
for df in dataframes:
    df['predicted_numeric'] = df['predicted'].apply(lambda x: 1 if x == 'LABEL_1' else 0)

# Calculate MCC score for individual models
mcc_scores = {}
for i, (df, model_name) in enumerate(zip(dataframes, model_names)):
    mcc = matthews_corrcoef(df['target'], df['predicted_numeric'])
    mcc_scores[model_name] = mcc

# Separate MCC scores by model type (GPT and BERT)
gpt_mcc_scores = [mcc_scores[model_name] for model_name in model_names if 'gpt' in model_name]
bert_mcc_scores = [mcc_scores[model_name] for model_name in model_names if 'bert' in model_name]

# Calculate Mean (STD), Max, and Ensemble MCC for GPT and BERT models
gpt_mean_mcc = np.mean(gpt_mcc_scores)
gpt_std_mcc = np.std(gpt_mcc_scores)
gpt_max_mcc = np.max(gpt_mcc_scores)

bert_mean_mcc = np.mean(bert_mcc_scores)
bert_std_mcc = np.std(bert_mcc_scores)
bert_max_mcc = np.max(bert_mcc_scores)

# Create a dataframe for ensemble calculations
# Drop redundant 'textid' and 'target' from subsequent DataFrames
ensemble_df = pd.concat(
    [dataframes[0][['textid', 'target']]] + [df[['predicted_numeric']] for df in dataframes],
    axis=1
)

# Rename columns appropriately for easier identification
ensemble_df.columns = ['textid', 'target'] + [f'{model_name}_pred' for model_name in model_names]

# Majority Voting Ensemble for GPT and BERT separately
gpt_predictions = [f'{model_name}_pred' for model_name in model_names if 'gpt' in model_name]
bert_predictions = [f'{model_name}_pred' for model_name in model_names if 'bert' in model_name]

ensemble_df['gpt_majority_pred'] = (ensemble_df[gpt_predictions].sum(axis=1) >= 2).astype(int)  # 2/3 models agree
ensemble_df['bert_majority_pred'] = (ensemble_df[bert_predictions].sum(axis=1) >= 2).astype(int)  # 2/3 models agree

# Calculate Ensemble MCC for GPT and BERT
gpt_ensemble_mcc = matthews_corrcoef(ensemble_df['target'], ensemble_df['gpt_majority_pred'])
bert_ensemble_mcc = matthews_corrcoef(ensemble_df['target'], ensemble_df['bert_majority_pred'])

# Prepare data for the table in a Pandas DataFrame
data = {
    "Model": ["GPT", "BERT"],
    "Mean (STD)": [
        f"{gpt_mean_mcc:.3f} ({gpt_std_mcc:.3f})",
        f"{bert_mean_mcc:.3f} ({bert_std_mcc:.3f})"
    ],
    "Max": [
        f"{gpt_max_mcc:.3f}",
        f"{bert_max_mcc:.3f}"
    ],
    "Ensemble": [
        f"{gpt_ensemble_mcc:.3f}",
        f"{bert_ensemble_mcc:.3f}"
    ]
}

# Create DataFrame
df = pd.DataFrame(data)

# Display the DataFrame
print(df)

# Convert the DataFrame to LaTeX code
latex_table = df.to_latex(index=False, column_format='lccc', bold_rows=True)

# Display the LaTeX code
print("\nLaTeX Table:\n")
print(latex_table)

print('done')
